##Baseline Method

In [ ]:
import os, cv2, torch, numpy as np
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.utils as vutils
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from sklearn.metrics import mutual_info_score

# CONFIGURATION

BASE_PATH = "/kaggle/input/datasets/mdrafsanyeasir/medical-dataset45-thesis/final_preprocessed_dataset"
MODALITY = "CT-MRI"       
EPOCHS =5
BATCH_SIZE = 4
LR = 1e-4
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# DATASET CLASS

class FusionDataset(Dataset):
    def __init__(self, base_path, split, modality):
        mod1, mod2 = modality.split("-")
        self.path1 = os.path.join(base_path, split, modality, mod1)
        self.path2 = os.path.join(base_path, split, modality, mod2)
        self.imgs = sorted(os.listdir(self.path1))

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, idx):
        img1 = cv2.imread(os.path.join(self.path1, self.imgs[idx]))
        img2 = cv2.imread(os.path.join(self.path2, self.imgs[idx]))

        img1 = cv2.resize(img1, (256,256)) / 255.0
        img2 = cv2.resize(img2, (256,256)) / 255.0

        img1 = torch.tensor(img1).permute(2,0,1).float()
        img2 = torch.tensor(img2).permute(2,0,1).float()

        return img1, img2


# MODEL ARCHITECTURE

class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3,64,3,padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64,128,3,padding=1), nn.ReLU(),
            nn.MaxPool2d(2)
        )

    def forward(self,x):
        return self.net(x)

class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(128,64,2,stride=2), nn.ReLU(),
            nn.ConvTranspose2d(64,3,2,stride=2),
            nn.Sigmoid()
        )

    def forward(self,x):
        return self.net(x)

class FusionNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = Encoder()
        self.decoder = Decoder()

    def forward(self, x1, x2):
        f1 = self.encoder(x1)
        f2 = self.encoder(x2)
        fused = (f1 + f2) / 2   # Average fusion rule
        return self.decoder(fused)


# TRAINING

train_dataset = FusionDataset(BASE_PATH, "train", MODALITY)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

model = FusionNet().to(DEVICE)
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=LR)

print("Training Started...")
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0

    for img1, img2 in train_loader:
        img1, img2 = img1.to(DEVICE), img2.to(DEVICE)

        fused = model(img1, img2)
        loss = criterion(fused, img1) + criterion(fused, img2)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch [{epoch+1}/{EPOCHS}]  Loss: {total_loss/len(train_loader):.4f}")


# SAVE FUSED IMAGES (TEST)

os.makedirs(f"fused_outputs/{MODALITY}", exist_ok=True)

test_dataset = FusionDataset(BASE_PATH, "test", MODALITY)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

model.eval()
print("Generating fused images...")
with torch.no_grad():
    for i, (img1, img2) in enumerate(test_loader):
        img1, img2 = img1.to(DEVICE), img2.to(DEVICE)
        fused = model(img1, img2)
        vutils.save_image(fused, f"fused_outputs/{MODALITY}/fused_{i}.png")


# EVALUATION METRICS (FIXED)

def compute_metrics(img1, img2, fused):
    img1 = img1.cpu().numpy().transpose(1,2,0)
    img2 = img2.cpu().numpy().transpose(1,2,0)
    fused = fused.cpu().numpy().transpose(1,2,0)

    psnr = (peak_signal_noise_ratio(img1, fused, data_range=1.0) +
            peak_signal_noise_ratio(img2, fused, data_range=1.0)) / 2

    ssim = (structural_similarity(img1, fused, channel_axis=2, data_range=1.0) +
            structural_similarity(img2, fused, channel_axis=2, data_range=1.0)) / 2

    mi = (mutual_info_score(img1[:,:,0].ravel(), fused[:,:,0].ravel()) +
          mutual_info_score(img2[:,:,0].ravel(), fused[:,:,0].ravel())) / 2

    return psnr, ssim, mi


# METRIC CALCULATION

psnr_list, ssim_list, mi_list = [], [], []

with torch.no_grad():
    for img1, img2 in test_loader:
        img1, img2 = img1.to(DEVICE), img2.to(DEVICE)
        fused = model(img1, img2)

        psnr, ssim, mi = compute_metrics(img1[0], img2[0], fused[0])
        psnr_list.append(psnr)
        ssim_list.append(ssim)
        mi_list.append(mi)

print("\nFINAL RESULTS")
print("Average PSNR:", np.mean(psnr_list))
print("Average SSIM:", np.mean(ssim_list))
print("Average MI  :", np.mean(mi_list))

CT-MRI VGG19

In [ ]:
# MULTIMODAL IMAGE FUSION (VGG19 CT-MRI)

import os, cv2, torch, numpy as np
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from sklearn.metrics import mutual_info_score
import torchvision.utils as vutils

# CONFIG

BASE_PATH = "/kaggle/input/datasets/mdrafsanyeasir/medical-dataset45-thesis/final_preprocessed_dataset"
MODALITY = "CT-MRI"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

EPOCHS = 15
BATCH_SIZE = 4
LR = 1e-4

# DATASET (AUTO FIXED)

class FusionDataset(Dataset):
    def __init__(self, base_path, split, modality):
        m1, m2 = modality.split("-")

        p1 = os.path.join(base_path, split, modality, m1)
        p2 = os.path.join(base_path, split, modality, m2)

        if not os.path.exists(p1):
            p1 = os.path.join(base_path, split, m1)
            p2 = os.path.join(base_path, split, m2)

        self.p1, self.p2 = p1, p2
        self.names = sorted(os.listdir(self.p1))

    def __len__(self): return len(self.names)

    def __getitem__(self, idx):
        ct = cv2.imread(os.path.join(self.p1, self.names[idx]))
        mri = cv2.imread(os.path.join(self.p2, self.names[idx]))

        ct = cv2.resize(ct,(256,256))/255.0
        mri = cv2.resize(mri,(256,256))/255.0

        ct = torch.tensor(ct).permute(2,0,1).float()
        mri = torch.tensor(mri).permute(2,0,1).float()

        return ct, mri

train_loader = DataLoader(FusionDataset(BASE_PATH,"train",MODALITY),
                          batch_size=BATCH_SIZE, shuffle=True)

test_dataset = FusionDataset(BASE_PATH,"test",MODALITY)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

# MODEL

class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3,64,3,padding=1), nn.ReLU(),
            nn.Conv2d(64,128,3,padding=1), nn.ReLU(),
            nn.MaxPool2d(2)
        )
    def forward(self,x): return self.net(x)

class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(128,64,2,stride=2), nn.ReLU(),
            nn.Conv2d(64,3,3,padding=1),
            nn.Sigmoid()
        )
    def forward(self,x): return self.net(x)

class FusionNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = Encoder()
        self.dec = Decoder()

    def forward(self, ct, mri):
        f1 = self.enc(ct)
        f2 = self.enc(mri)

       # FUSION
        fused = 0.5*(f1 + f2) + 0.5*torch.max(f1, f2)

        out = self.dec(fused)

        #SHARPENING
        blur = F.avg_pool2d(out,3,1,1)
        out = out + 0.2*(out - blur)

        return out

model = FusionNet().to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LR)
mse = nn.MSELoss()

# SSIM + EDGE LOSS

def ssim_loss(x,y):
    x = x.detach().permute(0,2,3,1).cpu().numpy()
    y = y.detach().permute(0,2,3,1).cpu().numpy()
    return 1 - np.mean([
        structural_similarity(x[i],y[i],channel_axis=2,data_range=1.0)
        for i in range(x.shape[0])
    ])

def edge_loss(x,y):
    sobel = torch.tensor([[1,0,-1],[2,0,-2],[1,0,-1]],
                         dtype=torch.float32,device=x.device).view(1,1,3,3)
    def grad(img):
        g = img.mean(dim=1,keepdim=True)
        return F.conv2d(g,sobel,padding=1)
    return torch.mean((grad(x)-grad(y))**2)

# TRAIN

print("Training Improved Model...")
for e in range(EPOCHS):
    model.train()
    total = 0

    for ct,mri in train_loader:
        ct,mri = ct.to(DEVICE), mri.to(DEVICE)
        fused = model(ct,mri)

        loss = (
            3.5*mse(fused,mri) +
            0.5*mse(fused,ct) +
            3.0*ssim_loss(fused,mri) +
            1.0*edge_loss(fused,mri)
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total += loss.item()

    print(f"Epoch {e+1}/{EPOCHS} Loss {total/len(train_loader):.4f}")

# TEST + METRICS

psnr_list, ssim_list, mi_list = [], [], []

def metrics(a,b):
    return (
        peak_signal_noise_ratio(a,b,data_range=1.0),
        structural_similarity(a,b,channel_axis=2,data_range=1.0),
        mutual_info_score((a[:,:,0]*255).astype(int).ravel(),
                          (b[:,:,0]*255).astype(int).ravel())
    )

model.eval()
with torch.no_grad():
    for ct_t,mri_t in test_loader:
        f = model(ct_t.to(DEVICE),mri_t.to(DEVICE))[0]
        f = f.cpu().numpy().transpose(1,2,0)

        mri = mri_t[0].numpy().transpose(1,2,0)

        p,s,m = metrics(mri,f)

        psnr_list.append(p)
        ssim_list.append(s)
        mi_list.append(m)

print("\nFINAL RESULTS (IMPROVED)")
print("PSNR:", np.mean(psnr_list))
print("SSIM:", np.mean(ssim_list))
print("MI  :", np.mean(mi_list))

## PET MRI

In [ ]:
#MULTIMODAL IMAGE FUSION (PET-MRI)  

import os, cv2, torch, numpy as np
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from sklearn.metrics import mutual_info_score

# CONFIG

BASE_PATH = "/kaggle/input/datasets/mdrafsanyeasir/medical-dataset45-thesis/final_preprocessed_dataset"
MODALITY = "PET-MRI"  
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

EPOCHS = 15
BATCH_SIZE = 4
LR = 1e-4

# DATASET

class FusionDataset(Dataset):
    def __init__(self, base_path, split, modality):
        m1, m2 = modality.split("-")

        p1 = os.path.join(base_path, split, modality, m1)
        p2 = os.path.join(base_path, split, modality, m2)

        if not os.path.exists(p1):
            p1 = os.path.join(base_path, split, m1)
            p2 = os.path.join(base_path, split, m2)

        self.p1, self.p2 = p1, p2
        self.names = sorted(os.listdir(self.p1))

    def __len__(self): return len(self.names)

    def __getitem__(self, idx):
        pet = cv2.imread(os.path.join(self.p1, self.names[idx]))   
        
        mri = cv2.imread(os.path.join(self.p2, self.names[idx]))

        pet = cv2.resize(pet,(256,256))/255.0
        mri = cv2.resize(mri,(256,256))/255.0

        pet = torch.tensor(pet).permute(2,0,1).float()
        mri = torch.tensor(mri).permute(2,0,1).float()

        return pet, mri

train_loader = DataLoader(FusionDataset(BASE_PATH,"train",MODALITY),
                          batch_size=BATCH_SIZE, shuffle=True)

test_dataset = FusionDataset(BASE_PATH,"test",MODALITY)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

# MODEL

class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3,64,3,padding=1), nn.ReLU(),
            nn.Conv2d(64,128,3,padding=1), nn.ReLU(),
            nn.MaxPool2d(2)
        )
    def forward(self,x): return self.net(x)

class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(128,64,2,2), nn.ReLU(),
            nn.Conv2d(64,3,3,padding=1),
            nn.Sigmoid()
        )
    def forward(self,x): return self.net(x)

class FusionNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = Encoder()
        self.dec = Decoder()

    def forward(self, pet, mri):
        f1 = self.enc(pet)
        f2 = self.enc(mri)

        fused = 0.5*(f1 + f2) + 0.5*torch.max(f1, f2)

        out = self.dec(fused)

        blur = F.avg_pool2d(out,3,1,1)
        out = out + 0.2*(out - blur)

        return out

model = FusionNet().to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LR)
mse = nn.MSELoss()

# LOSS FUNCTIONS

def ssim_loss(x,y):
    x = x.detach().permute(0,2,3,1).cpu().numpy()
    y = y.detach().permute(0,2,3,1).cpu().numpy()
    return 1 - np.mean([
        structural_similarity(x[i],y[i],channel_axis=2,data_range=1.0)
        for i in range(x.shape[0])
    ])

def edge_loss(x,y):
    sobel = torch.tensor([[1,0,-1],[2,0,-2],[1,0,-1]],
                         dtype=torch.float32,device=x.device).view(1,1,3,3)
    def grad(img):
        g = img.mean(dim=1,keepdim=True)
        return F.conv2d(g,sobel,padding=1)
    return torch.mean((grad(x)-grad(y))**2)

# TRAIN

print("Training PET-MRI Model...")
for e in range(EPOCHS):
    model.train()
    total = 0

    for pet,mri in train_loader:
        pet,mri = pet.to(DEVICE), mri.to(DEVICE)
        fused = model(pet,mri)

        loss = (
            3.5*mse(fused,mri) +
            0.5*mse(fused,pet) +
            3.0*ssim_loss(fused,mri) +
            1.0*edge_loss(fused,mri)
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total += loss.item()

    print(f"Epoch {e+1}/{EPOCHS} Loss {total/len(train_loader):.4f}")

# TEST

psnr_list, ssim_list, mi_list = [], [], []

def metrics(a,b):
    return (
        peak_signal_noise_ratio(a,b,data_range=1.0),
        structural_similarity(a,b,channel_axis=2,data_range=1.0),
        mutual_info_score((a[:,:,0]*255).astype(int).ravel(),
                          (b[:,:,0]*255).astype(int).ravel())
    )

model.eval()
with torch.no_grad():
    for pet_t,mri_t in test_loader:
        f = model(pet_t.to(DEVICE),mri_t.to(DEVICE))[0]
        f = f.cpu().numpy().transpose(1,2,0)

        mri = mri_t[0].numpy().transpose(1,2,0)

        p,s,m = metrics(mri,f)

        psnr_list.append(p)
        ssim_list.append(s)
        mi_list.append(m)

print("\nFINAL RESULTS (PET-MRI)")
print("PSNR:", np.mean(psnr_list))
print("SSIM:", np.mean(ssim_list))
print("MI  :", np.mean(mi_list))

## Spect Mri

In [ ]:
# FINAL IMPROVED MULTIMODAL IMAGE FUSION (SPECT-MRI)

import os, cv2, torch, numpy as np
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from skimage.metrics import peak_signal_noise_ratio, structural_similarity
from sklearn.metrics import mutual_info_score

# CONFIG

BASE_PATH = "/kaggle/input/datasets/mdrafsanyeasir/medical-dataset45-thesis/final_preprocessed_dataset"
MODALITY = "SPECT-MRI" 
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

EPOCHS = 15
BATCH_SIZE = 4
LR = 1e-4

# DATASET

class FusionDataset(Dataset):
    def __init__(self, base_path, split, modality):
        m1, m2 = modality.split("-")

        p1 = os.path.join(base_path, split, modality, m1)
        p2 = os.path.join(base_path, split, modality, m2)

        if not os.path.exists(p1):
            p1 = os.path.join(base_path, split, m1)
            p2 = os.path.join(base_path, split, m2)

        self.p1, self.p2 = p1, p2
        self.names = sorted(os.listdir(self.p1))

    def __len__(self): return len(self.names)

    def __getitem__(self, idx):
        pet = cv2.imread(os.path.join(self.p1, self.names[idx]))  
        mri = cv2.imread(os.path.join(self.p2, self.names[idx]))

        pet = cv2.resize(pet,(256,256))/255.0
        mri = cv2.resize(mri,(256,256))/255.0

        pet = torch.tensor(pet).permute(2,0,1).float()
        mri = torch.tensor(mri).permute(2,0,1).float()

        return pet, mri

train_loader = DataLoader(FusionDataset(BASE_PATH,"train",MODALITY),
                          batch_size=BATCH_SIZE, shuffle=True)

test_dataset = FusionDataset(BASE_PATH,"test",MODALITY)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

# MODEL

class Encoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3,64,3,padding=1), nn.ReLU(),
            nn.Conv2d(64,128,3,padding=1), nn.ReLU(),
            nn.MaxPool2d(2)
        )
    def forward(self,x): return self.net(x)

class Decoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(128,64,2,2), nn.ReLU(),
            nn.Conv2d(64,3,3,padding=1),
            nn.Sigmoid()
        )
    def forward(self,x): return self.net(x)

class FusionNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc = Encoder()
        self.dec = Decoder()

    def forward(self, pet, mri):
        f1 = self.enc(pet)
        f2 = self.enc(mri)

        fused = 0.5*(f1 + f2) + 0.5*torch.max(f1, f2)

        out = self.dec(fused)

        blur = F.avg_pool2d(out,3,1,1)
        out = out + 0.2*(out - blur)

        return out

model = FusionNet().to(DEVICE)
optimizer = optim.Adam(model.parameters(), lr=LR)
mse = nn.MSELoss()

# LOSS FUNCTIONS

def ssim_loss(x,y):
    x = x.detach().permute(0,2,3,1).cpu().numpy()
    y = y.detach().permute(0,2,3,1).cpu().numpy()
    return 1 - np.mean([
        structural_similarity(x[i],y[i],channel_axis=2,data_range=1.0)
        for i in range(x.shape[0])
    ])

def edge_loss(x,y):
    sobel = torch.tensor([[1,0,-1],[2,0,-2],[1,0,-1]],
                         dtype=torch.float32,device=x.device).view(1,1,3,3)
    def grad(img):
        g = img.mean(dim=1,keepdim=True)
        return F.conv2d(g,sobel,padding=1)
    return torch.mean((grad(x)-grad(y))**2)

# TRAIN

print("Training SPECT-MRI Model...")
for e in range(EPOCHS):
    model.train()
    total = 0

    for pet,mri in train_loader:
        pet,mri = pet.to(DEVICE), mri.to(DEVICE)
        fused = model(pet,mri)

        loss = (
            3.5*mse(fused,mri) +
            0.5*mse(fused,pet) +
            3.0*ssim_loss(fused,mri) +
            1.0*edge_loss(fused,mri)
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total += loss.item()

    print(f"Epoch {e+1}/{EPOCHS} Loss {total/len(train_loader):.4f}")

# TEST

psnr_list, ssim_list, mi_list = [], [], []

def metrics(a,b):
    return (
        peak_signal_noise_ratio(a,b,data_range=1.0),
        structural_similarity(a,b,channel_axis=2,data_range=1.0),
        mutual_info_score((a[:,:,0]*255).astype(int).ravel(),
                          (b[:,:,0]*255).astype(int).ravel())
    )

model.eval()
with torch.no_grad():
    for pet_t,mri_t in test_loader:
        f = model(pet_t.to(DEVICE),mri_t.to(DEVICE))[0]
        f = f.cpu().numpy().transpose(1,2,0)

        mri = mri_t[0].numpy().transpose(1,2,0)

        p,s,m = metrics(mri,f)

        psnr_list.append(p)
        ssim_list.append(s)
        mi_list.append(m)

print("\nFINAL RESULTS (PET-MRI)")
print("PSNR:", np.mean(psnr_list))
print("SSIM:", np.mean(ssim_list))
print("MI  :", np.mean(mi_list))